# Channel Correlation Matrix

**Dataset**: BNCI2014-001 (Motor Imagery)
**Subject**: 1
**Paradigm**: MotorImagery (n_classes=2)
**Channels**: 22 EEG
**Sampling rate**: 250 Hz

---

## Overview

This notebook computes the average Pearson correlation matrix across all 22 EEG channels of BNCI2014-001, averaged over all trials, and visualizes it as an interactive heatmap.

## What this notebook does

- Loads BNCI2014-001 subject 1 via MOABB
- Extracts epochs with the MotorImagery paradigm
- Computes per-trial correlation matrices with np.corrcoef
- Averages across all trials and plots the result as a heatmap

## What you should expect to see

- A 22x22 correlation matrix with values from -1 to +1
- High correlation on the diagonal (each channel with itself)
- Neighboring channels (e.g., C3-C1, Cz-C2) show higher correlation
- EOG-free EEG channels reveal spatial connectivity patterns

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| dataset | BNCI2014_001 | MOABB motor imagery dataset |
| subjects | [1] | Subject 1 only |
| n_classes | 2 | MotorImagery paradigm classes |
| method | Pearson | Correlation type |
| n_channels | 22 | EEG channels in epochs |



## 1. Install dependencies


In [ ]:
!pip install moabb mne scipy numpy plotly


## 2. Load MOABB dataset

MOABB downloads data automatically on first use (~44 MB for subject 1). Subsequent runs use cached data.



In [ ]:
from moabb.datasets import BNCI2014_001
ds = BNCI2014_001()
sessions = ds.get_data(subjects=[1])
subject_key = list(sessions.keys())[0]
session_dict = sessions[subject_key]
n_sessions = len(session_dict)
n_runs = len(next(iter(session_dict.values())))
first_run = next(iter(next(iter(session_dict.values())).values()))
n_channels_raw = len(first_run.ch_names)
sfreq = first_run.info['sfreq']
print(f'Subject 1: {n_sessions} sessions, {n_runs} runs/session')
print(f'Raw channels: {n_channels_raw}, Sampling rate: {sfreq} Hz')
print(f'Channel names: {first_run.ch_names}')



In [ ]:
from moabb.paradigms import MotorImagery
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=ds, subjects=[1])
print(f'X shape: {X.shape}  (n_trials, n_channels, n_samples)')
print(f'Labels shape: {labels.shape}')
print(f'Meta shape: {meta.shape}')



## 3. Explore the data

We print the epoch shapes and channel information.


In [ ]:
import numpy as np
unique_labels, counts = np.unique(labels, return_counts=True)
print(f'Epoch channels: {X.shape[1]}')
print(f'Epoch samples: {X.shape[2]}')
print(f'Epoch duration: {X.shape[2] / sfreq:.2f} s')
print(f'Unique labels: {list(unique_labels)}')
print(f'Trials per class: {dict(zip(unique_labels, counts))}')
print(f'Total trials: {X.shape[0]}')



## 4. Apply the analysis

We compute the per-trial correlation matrix and average across all trials.



In [ ]:
import numpy as np
n_trials, n_channels, n_samples = X.shape
corr_sum = np.zeros((n_channels, n_channels))
for trial in range(n_trials):
    trial_data = X[trial]
    std = np.std(trial_data, axis=1, keepdims=True)
    std[std == 0] = 1.0
    normed = (trial_data - np.mean(trial_data, axis=1, keepdims=True)) / std
    corr_sum += np.corrcoef(normed)
corr_matrix = corr_sum / n_trials
print(f'Correlation matrix shape: {corr_matrix.shape}')
print(f'Mean off-diagonal correlation: {np.mean(corr_matrix[~np.eye(n_channels, dtype=bool)]):.3f}')



## 5. Interactive plot

**What to look for:**

- The diagonal is perfectly correlated (value = 1.0)
- Neighboring electrodes show higher correlation (warm colors)
- Distant electrodes show lower or negative correlation (cool colors)
- The matrix is symmetric around the diagonal




In [ ]:
import plotly.graph_objects as go
raw_ch = first_run.ch_names
eog_idx = [i for i, n in enumerate(raw_ch) if n.startswith('EOG') or n == 'STI']
ch_labels = [n for i, n in enumerate(raw_ch) if i not in eog_idx]
fig = go.Figure(data=go.Heatmap(z=corr_matrix, x=ch_labels, y=ch_labels,
                                 colorscale='RdBu', zmin=-1, zmax=1))
fig.update_layout(title='Channel Correlation Matrix - BNCI2014-001',
                  xaxis_title='Channel', yaxis_title='Channel',
                  width=700, height=700)
fig.show()



## What did we learn?

- The correlation matrix reveals spatial relationships between EEG channels
- Neighboring electrodes are more correlated than distant ones
- Averaging across trials gives a stable estimate of channel connectivity
- This matrix is useful for feature selection and channel reduction


